# Data Science Lab 10
## Feature Engineering and Feature Selection
### UET Peshawar

## Cell 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, PolynomialFeatures
from sklearn.feature_selection import SelectKBest, chi2, RFE
from sklearn.linear_model import LogisticRegression, Lasso

## Cell 2: Load Dataset

In [ ]:
df=pd.read_excel('Feature_Engineering_Dataset_100Rows.xlsx')

## Cell 3: Display First 10 Records

In [ ]:
df.head(10)

## Cell 4: Dataset Shape

In [ ]:
print(df.shape)

## Cell 5: Dataset Information

In [ ]:
df.info()

## Cell 6: Statistical Summary

In [ ]:
df.describe()

# Task 1

## Cell 7: Original Income

In [ ]:
df["Family_Income"].head(10)

## Cell 8: Apply Log Transformation

In [ ]:
df["Log_Income"]=np.log(df["Family_Income"]+1)

## Cell 9: Compare Original and Transformed

In [ ]:
df[["Family_Income","Log_Income"]].head(10)

## Cell 10: Answers

In [ ]:
print("Q1: Reduces skewness.\nQ2: Compresses large values.")

# Task 2

## Cell 11: Create Encoders

In [ ]:
le_gender=LabelEncoder()
le_result=LabelEncoder()

## Cell 12: Encode Gender

In [ ]:
df["Gender_Encoded"]=le_gender.fit_transform(df["Gender"])

## Cell 13: Encode Final_Result

In [ ]:
df["Result_Encoded"]=le_result.fit_transform(df["Final_Result"])

## Cell 14: Display Encoded Columns

In [ ]:
df[["Gender","Gender_Encoded","Final_Result","Result_Encoded"]].head(10)

## Cell 15: Encoding Mapping

In [ ]:
print(dict(zip(le_gender.classes_,le_gender.transform(le_gender.classes_))))
print(dict(zip(le_result.classes_,le_result.transform(le_result.classes_))))

# Task 3

## Cell 16: One Hot Encode Department

In [ ]:
dept=pd.get_dummies(df["Department"],prefix="Department")
df=pd.concat([df,dept],axis=1)

## Cell 17: Display New Columns

In [ ]:
df.filter(like="Department").head(10)

## Cell 18: Answers

In [ ]:
print("One-Hot treats each category independently.")
print("Label Encoding creates false ordering.")

# Task 4

## Cell 19: Create Interaction Feature

In [ ]:
df["Study_Attendance"]=df["Study_Hours"]*df["Attendance"]

## Cell 20: Display Feature

In [ ]:
df[["Study_Hours","Attendance","Study_Attendance"]].head(10)

## Cell 21: Answers

In [ ]:
print("Interaction combines two variables.")
print("Improves model learning.")

# Task 5

## Cell 22: Create Polynomial Object

In [ ]:
poly=PolynomialFeatures(degree=2,include_bias=False)

## Cell 23: Generate Features

In [ ]:
poly_features=poly.fit_transform(df[["Study_Hours","Previous_GPA"]])

## Cell 24: Convert to DataFrame

In [ ]:
poly_df=pd.DataFrame(poly_features,columns=poly.get_feature_names_out(["Study_Hours","Previous_GPA"]))
poly_df.head()

## Cell 25: Add Required Columns

In [ ]:
df["Study_Hours2"]=poly_df["Study_Hours^2"]
df["Previous_GPA2"]=poly_df["Previous_GPA^2"]
df["Study_Hours_Previous_GPA"]=poly_df["Study_Hours Previous_GPA"]

## Cell 26: Display Features

In [ ]:
df[["Study_Hours2","Previous_GPA2","Study_Hours_Previous_GPA"]].head(10)

## Cell 27: Answers

In [ ]:
print("Polynomial features capture nonlinear relationships.")

# Part B

## Cell 28: Copy Dataset

In [ ]:
model_df=df.copy()

## Cell 29: Encode Remaining Columns

In [ ]:
model_df["Gender"]=LabelEncoder().fit_transform(model_df["Gender"])
model_df["Department"]=LabelEncoder().fit_transform(model_df["Department"])
model_df["Final_Result"]=LabelEncoder().fit_transform(model_df["Final_Result"])

## Cell 30: Separate X and y

In [ ]:
X=model_df.drop("Final_Result",axis=1)
y=model_df["Final_Result"]

# Task 6

## Cell 31: Apply Chi Square

In [ ]:
selector=SelectKBest(chi2,k=3)
selector.fit(X,y)

## Cell 32: Scores

In [ ]:
scores=pd.DataFrame({"Feature":X.columns,"Score":selector.scores_})
scores.sort_values("Score",ascending=False)

## Cell 33: Selected Features

In [ ]:
print(X.columns[selector.get_support()])

## Cell 34: Answers

In [ ]:
print("Highest score = Most important feature.")
print("Filter method selects before training.")

# Task 7

## Cell 35: Train Logistic Regression

In [ ]:
model=LogisticRegression(max_iter=1000)

## Cell 36: Apply RFE

In [ ]:
rfe=RFE(model,n_features_to_select=3)
rfe.fit(X,y)

## Cell 37: Rankings

In [ ]:
pd.DataFrame({"Feature":X.columns,"Ranking":rfe.ranking_,"Selected":rfe.support_})

## Cell 38: Answers

In [ ]:
print("Ranking=1 means selected.")

# Task 8

## Cell 39: Train LASSO

In [ ]:
lasso=Lasso(alpha=0.01)
lasso.fit(X,y)

## Cell 40: Coefficients

In [ ]:
coef=pd.DataFrame({"Feature":X.columns,"Coefficient":lasso.coef_})
coef

## Cell 41: Removed Features

In [ ]:
coef[coef.Coefficient==0]

## Cell 42: Retained Features

In [ ]:
coef[coef.Coefficient!=0]

## Cell 43: Answers

In [ ]:
print("Zero coefficient features are removed.")
print("LASSO shrinks coefficients to zero automatically.")

## Cell 44: Save Dataset

In [ ]:
df.to_excel("Feature_Engineering_Output.xlsx",index=False)
print("Output Saved Successfully")